# 1. Data Preparation & Tokenizer Training

Notebook này được thiết kế để chạy trên **Google Colab**.
Mục đích: Tải dữ liệu từ HuggingFace (dùng tập `opus100`), làm sạch, huấn luyện Tokenizer và lưu toàn bộ kết quả thẳng vào **Google Drive** của bạn.

*(Lưu ý: Tập opus100 chứa 1 triệu câu cho mỗi cặp ngôn ngữ, đây là số lượng cực kỳ lý tưởng để train Transformer Base!)*

In [1]:
!pip install datasets tokenizers sacremoses pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 22.5 MB/s eta 0:00:00


In [6]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Tạo thư mục gốc trên Drive
BASE_DIR = '/content/drive/MyDrive/Multilingual_MT'
os.makedirs(f'{BASE_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{BASE_DIR}/tokenizer', exist_ok=True)
os.makedirs(f'{BASE_DIR}/model_assets', exist_ok=True)

print(f"Đã tạo/kết nối thư mục gốc tại: {BASE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tạo/kết nối thư mục gốc tại: /content/drive/MyDrive/Multilingual_MT


In [8]:
from datasets import load_dataset
import random

langs = ['en-vi', 'en-ja', 'en-zh']

train_file = f"{BASE_DIR}/data/processed/train.txt"
test_file = f"{BASE_DIR}/data/processed/test.txt"

# Hàm tạo tag ngôn ngữ đích
def get_lang_tag(lang_code):
    mapping = {'vi': '<2vi>', 'ja': '<2ja>', 'zh': '<2zh>', 'en': '<2en>'}
    return mapping.get(lang_code, '')

print("Bắt đầu tải và lưu dữ liệu (khoảng 3 triệu câu gốc -> 6 triệu dòng training do lưu cả 2 chiều)...")

with open(train_file, 'w', encoding='utf-8') as f_train, \
     open(test_file, 'w', encoding='utf-8') as f_test:

    for pair in langs:
        print(f"\nĐang tải và xử lý cặp: {pair}...")
        dataset = load_dataset("Helsinki-NLP/opus-100", pair)

        src_code, tgt_code = pair.split('-')
        src_tag = get_lang_tag(src_code)
        tgt_tag = get_lang_tag(tgt_code)

        # Tập Train: ghi cả 2 chiều EN->TGT và TGT->EN
        count = 0
        for row in dataset['train']:
            text_src = row['translation'][src_code].replace('\n', ' ').strip()
            text_tgt = row['translation'][tgt_code].replace('\n', ' ').strip()
            if not text_src or not text_tgt: continue

            f_train.write(f"{tgt_tag}\t{text_src}\t{text_tgt}\n")
            f_train.write(f"{src_tag}\t{text_tgt}\t{text_src}\n")
            count += 2
            if count % 200000 == 0:
                print(f"  Đã xử lý {count} dòng...")

        # Tập Test: tương tự
        for row in dataset['test']:
            text_src = row['translation'][src_code].replace('\n', ' ').strip()
            text_tgt = row['translation'][tgt_code].replace('\n', ' ').strip()
            if not text_src or not text_tgt: continue
            f_test.write(f"{tgt_tag}\t{text_src}\t{text_tgt}\n")
            f_test.write(f"{src_tag}\t{text_tgt}\t{text_src}\n")

print(f"\nXong! File data đã lưu an toàn trên Google Drive tại {BASE_DIR}/data/processed/")

Bắt đầu tải và lưu dữ liệu (khoảng 3 triệu câu gốc -> 6 triệu dòng training do lưu cả 2 chiều)...

Đang tải và xử lý cặp: en-vi...


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-vi/test-00000-of-00001.parquet:   0%|          | 0.00/137k [00:00<?, ?B/s]

en-vi/train-00000-of-00001.parquet:   0%|          | 0.00/59.0M [00:00<?, ?B/s]

en-vi/validation-00000-of-00001.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

...

en-zh/validation-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  Đã xử lý 200000 dòng...
  Đã xử lý 400000 dòng...
  Đã xử lý 600000 dòng...
  Đã xử lý 800000 dòng...
  Đã xử lý 1000000 dòng...
  Đã xử lý 1200000 dòng...
  Đã xử lý 1400000 dòng...
  Đã xử lý 1600000 dòng...
  Đã xử lý 1800000 dòng...
  Đã xử lý 2000000 dòng...

Xong! File data đã lưu an toàn trên Google Drive tại /content/drive/MyDrive/Multilingual_MT/data/processed/


In [9]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("Bắt đầu huấn luyện Tokenizer (32k vocab, có thể tốn 5-10 phút)...")

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Coverage = 0.9995 quan trọng để lấy đủ ký tự Tiếng Nhật/Trung
trainer = BpeTrainer(
    vocab_size=32000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]", "<2vi>", "<2ja>", "<2zh>", "<2en>"],
    show_progress=True,
    initial_alphabet=[chr(i) for i in range(256)]
)

# Train trực tiếp từ file text Train khổng lồ trên Drive
tokenizer.train(files=[train_file], trainer=trainer)

tok_path = f"{BASE_DIR}/tokenizer/tokenizer.json"
tokenizer.save(tok_path)
print(f"Đã lưu Tokenizer thành công vào: {tok_path}")

# Test thử Tokenizer
output = tokenizer.encode("<2vi> Hello, how are you?")
print("Test Encode:", output.tokens)

Bắt đầu huấn luyện Tokenizer (32k vocab, có thể tốn 5-10 phút)...
Đã lưu Tokenizer thành công vào: /content/drive/MyDrive/Multilingual_MT/tokenizer/tokenizer.json
Test Encode: ['<2vi>', 'Hello', ',', 'how', 'are', 'you', '?']
